In [ ]:
import sys
sys.path.append('../src')
from circuit_postprocess import *
from should_be_stdlib import *
from neurodata import *
from circuits import *
from data import *

In [ ]:
from itertools import combinations_with_replacement
from typing import Callable

In [ ]:
from os import environ as ENV
ENV['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import jax
print(jax.devices())
jax.config.update('jax_enable_x64', True)

In [ ]:
!which catalyst

# pennylane-catalyst uv/venv metagarbage
import catalyst

# https://github.com/PennyLaneAI/catalyst/pull/1839
# catalyst.utils.runtime_environment.get_cli_path = lambda: ENV['PWD'] + '/.venv/bin/catalyst'
catalyst.utils.runtime_environment.get_cli_path()

In [ ]:
from tqdm.notebook import tqdm
import numpy as np
import pennylane as qml
from matplotlib import pyplot as plt
from seaborn import heatmap

In [ ]:
record = load_set()
tuning_curves = get_tc(record)
tuning_curves_rescaled = pd.read_csv(datapath('data_tuning-curves_rescaled.csv'), index_col=0)
tuning_curves_resampled = pd.read_csv(datapath('data_tuning-curves_resampled.csv'), index_col=0)

In [ ]:
dev_ang = qml.device('lightning.gpu', wires=18)  # GPU not that much slower, helps with indexing ig?
dev_amp = qml.device('lightning.qubit', wires=4) # CPU faster for 4 qubits
dev_ang, dev_amp

In [ ]:
# metrics['metric'] = (data, calculator for rows)
metrics:dict[str,tuple[object,Callable,object]] = {
    'ang': (
        tuning_curves_rescaled,
        lambda e, a, b: swap_expectation(e(a, b), 9)[1],
        executor(dev_ang)(circuit_angle_swap)
    ),
    'ang-qft': (
        tuning_curves_rescaled,
        lambda e, a, b: swap_expectation(e(a, b), 9)[1],
        executor(dev_ang)(circuit_angle_qft_swap)
    ),
    'amp': (
        tuning_curves_resampled,
        lambda e, a, b: e(a, b)[0],
        executor(dev_amp)(circuit_amp_iamp)
    ),
    'amp-qft': (
        tuning_curves_resampled,
        lambda e, a, b: e(a, b)[0],
        executor(dev_amp)(circuit_amp_iamp_qft)
    ),
}

In [ ]:
def get_fidelity_sub1(name_a_b: tuple[str, list[int], list[int]]) -> tuple[np.float64, np.float64]:
    name, a, b = name_a_b
    return [
        a, b,
        metrics[name][1](
            metrics[name][2],
            *small_then_big_array(
                metrics[name][0].loc[a].to_numpy(),
                metrics[name][0].loc[b].to_numpy(),
            )
        )
    ]

In [ ]:
def get_fidelity(name):
    pairs = combinations_with_replacement(tuning_curves.index, 2)
    pairs_len = len(tuning_curves) * (len(tuning_curves) + 1) // 2

    # from multiprocessing import Pool, cpu_count
    # with Pool(processes=cpu_count()) as pool:
    #     ab = list(tqdm(pool.imap(get_fidelity_sub1, [(name,a,b) for (a,b) in pairs]), total=pairs_len))
    ab = [get_fidelity_sub1((name,a,b)) for a,b in tqdm(pairs, total=pairs_len, desc=name)]

    return pd.DataFrame(
        ab,
        columns = ['A', 'B', 'fidelity']
    )

In [ ]:
for (name, _) in metrics.items():
    csv = datapath(f'results_simulator_{name}.csv')
    if not os.path.isfile(csv):
        f = get_fidelity(name).pivot_table(index='A', columns='B')['fidelity'].astype(np.float64)
        f.to_csv(csv)
    else:
        f = pd.read_csv(csv, index_col=0).astype(np.float64)

    plt.figure(figsize=(1,1), dpi=f.shape[0])
    ax = heatmap(mirror_matrix(f.to_numpy()), cbar=False, square=True)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.tight_layout(pad=0)
    plt.savefig(figspath(f'results_simulator_{name}.png'))
    plt.show()
    plt.close()